# Exploratory Data Analysis - Raw Project Data

**Purpose:** Understand data quality issues and inform cleaning strategy

**Date:** 2025-12-01

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')

## 1. Load Data

In [ ]:
df = pd.read_csv('../input/data_raw.csv')
print(f"Dataset shape: {df.shape}")
df.head(10)

## 2. Data Overview

In [ ]:
df.info()

In [ ]:
df.describe()

## 3. Missing Values Analysis

In [ ]:
missing = df.isna().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing': missing, 'Percentage': missing_pct})
print(missing_df[missing_df['Missing'] > 0])

In [ ]:
plt.figure(figsize=(10, 4))
sns.heatmap(df.isna(), cbar=False, yticklabels=False, cmap='viridis')
plt.title('Missing Data Pattern')
plt.tight_layout()
plt.savefig('../reports/missing_data_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

## 4. Column Analysis

### Budget_USD

In [ ]:
print(df['Budget_USD'].describe())
print(f"Missing: {df['Budget_USD'].isna().sum()}")

In [ ]:
plt.figure(figsize=(10, 4))
df['Budget_USD'].dropna().hist(bins=10, edgecolor='black')
plt.title('Budget Distribution')
plt.xlabel('Budget (USD)')
plt.ylabel('Frequency')
plt.tight_layout()
plt.savefig('../reports/budget_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

### Status

In [ ]:
print(df['Status'].value_counts(dropna=False))

In [ ]:
plt.figure(figsize=(8, 4))
df['Status'].value_counts(dropna=False).plot(kind='bar', edgecolor='black')
plt.title('Project Status Distribution')
plt.xlabel('Status')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('../reports/status_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

### Start_Date

In [ ]:
print(df['Start_Date'].value_counts(dropna=False))
invalid_dates = df['Start_Date'].apply(lambda x: pd.to_datetime(x, errors='coerce')).isna().sum()
print(f"\nInvalid dates: {invalid_dates}")

## 5. Data Quality Summary

In [ ]:
print("=== DATA QUALITY ISSUES ===")
print(f"Missing budgets: {df['Budget_USD'].isna().sum()}")
print(f"Missing status: {df['Status'].isna().sum()}")
print(f"Invalid dates: {df['Start_Date'].apply(lambda x: pd.to_datetime(x, errors='coerce')).isna().sum()}")
print(f"\nTotal rows: {len(df)}")
print(f"Rows with issues: {df.isna().any(axis=1).sum()}")

# Save summary to reports
quality_summary = {
    'Missing_Budgets': df['Budget_USD'].isna().sum(),
    'Missing_Status': df['Status'].isna().sum(),
    'Invalid_Dates': df['Start_Date'].apply(lambda x: pd.to_datetime(x, errors='coerce')).isna().sum(),
    'Total_Rows': len(df),
    'Rows_with_Issues': df.isna().any(axis=1).sum()
}
pd.DataFrame([quality_summary]).to_csv('../reports/data_quality_summary.csv', index=False)
print("\n✓ Saved data quality summary to reports/data_quality_summary.csv")

## 6. Recommendations

1. Standardize column names to lowercase snake_case
2. Fill missing budgets with 0.0
3. Standardize status to uppercase, fill missing with 'UNKNOWN'
4. Remove rows with unparseable dates
5. Add project_age_days feature